## Demo: Multifidelity UQ for OPF

Demo for running ACV-MRP on the OPF multifidelity model pair.

* HF model: ACOPF with intertemporal generator coupling.

* LF model: DCOPF (default) or copperplate dispatch with the same scenario batches.

The two models share:
- finite scenario population,
- one scenario sampler,
- one first-stage decision space.

In [1]:
from sparow.conf_intervals.options import UQOptions
from sparow.conf_intervals.acv_mrp import ACVMRP
from sparow.conf_intervals.evaluate_true_optimality_gap import TrueOptimalityGapEvaluator

# This is the function you have to define for your problem instance
from uq_opf import get_model_ensemble_for_uq

[    0.00] Initializing mpi-sppy
Alternative solutions package from or_topas is available.


In [2]:
# ------------------------------------------------------------------
# Step 1: Build the shared HF/LF ensemble
# ------------------------------------------------------------------
ensemble = get_model_ensemble_for_uq(
    model_name="HF",              # ignored; kept for interface consistency
    seed=12345,
    with_replacement=True,
    lf_model_type="dcopf",        # alternatives: "copperplate"
)

hf_model = ensemble.high_fidelity_model()
lf_model = ensemble.low_fidelity_model()

In [3]:
# ------------------------------------------------------------------
# Step 2: Generate one candidate first-stage solution xhat
# ------------------------------------------------------------------
# We first solve one HF SAA on a random subset of the finite scenario population and
# then extract the resulting first-stage vector as a candidate to evaluate.

n_xhat = 1 # choose a small batch size for candidate generation
xhat_replication_id = 999  # fixed id so the sampled batch is reproducible

xhat_scenarios = hf_model.draw_batch_of_scenarios(n=n_xhat, replication_id=xhat_replication_id,)

solved_hf = hf_model.solve_saa(
    sampled_scenarios=xhat_scenarios,
    solver_name="ipopt", # use nonlinear solver because ACOPF contains nonlinear, nonconvex expressions
    solver_options=None,
)

xhat = hf_model.get_first_stage_solution(solved_hf)

print("\nCandidate first-stage solution xhat extracted from one HF SAA on a random subset:")
print(f"Number of scenarios used to generate xhat: {n_xhat}")
for k, v in xhat.items():
    print(f"  {k}: {v}")

INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP



Candidate first-stage solution xhat extracted from one HF SAA on a random subset:
Number of scenarios used to generate xhat: 1
  time_periods[1].m.pg['1']: 2.1820366644023546
  time_periods[1].m.pg['2']: 0.7167423214444644
  time_periods[1].m.pg['3']: 0.0
  time_periods[1].m.pg['4']: 0.0
  time_periods[1].m.pg['5']: 0.0
  time_periods[1].m.pg['6']: 0.0


In [4]:
# ------------------------------------------------------------------
# Step 3: Configure ACV-MRP
# ------------------------------------------------------------------
options = UQOptions(
    n=4,                # batch size per replication
    m=10,                # paired HF/LF replications
    M=10,                # additional LF-only replications
    alpha=0.05,          # one-sided confidence level
    seed=12345,
    with_replacement=True,
    solver_name="ipopt",
    verbose=True,
)

In [5]:
# ------------------------------------------------------------------
# Step 4: Run ACV-MRP
# ------------------------------------------------------------------
acv_algorithm = ACVMRP(
    hf_model=hf_model,
    lf_model=lf_model,
    options=options,
)

results = acv_algorithm.run(xhat=xhat)

print("\nACV-MRP results:")
print(f"ACV-MRP Point Estimate: {results['point_estimate']}")
print(f"ACV-MRP Confidence Interval: [{results['ci_lower']}, {results['ci_upper']}]")
print(f"Estimated control variate coefficient: {results['control_variate_coefficient']}")
print(f"Estimated sample correlation: {results['sample_correlation']}")
print("\n")
print(f"Point estimate using only high-fidelity model (no paired or extra low-fidelity evals): {results['point_estimate_hf_only']}")
print(f"Variance reduction factor from spending additional computation on low-fidelity evals: {results['variance_reduction_factor']}")

INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).


Running ACV-MRP with m=10, M=10, n=4
Using precomputed superset of scenarios for nested sampling scheme: False
Running paired ACV-MRP replication 1/10


INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).


Gap estimate for high-fidelity paired replication 1: F_nk = 5236.056259545534
Gap estimate for low-fidelity paired replication 1 : G_nk = 4379.085257981162
Running paired ACV-MRP replication 2/10


INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).


...

Gap estimate for low-fidelity additional replication 18 : G_nk = 4396.454464371207
Running LF-only replication 9/10


INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).


Gap estimate for low-fidelity additional replication 19 : G_nk = 4410.833574631666
Running LF-only replication 10/10


INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP


Gap estimate for low-fidelity additional replication 20 : G_nk = 7084.602435440393

ACV-MRP results:
ACV-MRP Point Estimate: 5158.3790682555955
ACV-MRP Confidence Interval: [0.0, 6262.754272189982]
Estimated control variate coefficient: 0.959777059871869
Estimated sample correlation: 0.4961994477110763


Point estimate using only high-fidelity model (no paired or extra low-fidelity evals): 5348.5670216535245
Variance reduction factor from spending additional computation on low-fidelity evals: 11.403899202832381


In [6]:
# ------------------------------------------------------------------
# Step 5: Optional finite-population benchmark
# ------------------------------------------------------------------
# This computes the exact finite-population quantities over the stored
# scenario population for the HF model, which is useful for debugging
# and small-scale numerical validation.
true_gap_evaluator = TrueOptimalityGapEvaluator(
    model=hf_model,
    solver_name="ipopt",
    solver_options=None,
)

true_gap_results = true_gap_evaluator.compute_true_gap(xhat=xhat)

print("\nTrue finite-population HF quantities:")
print(f"True optimal value: {true_gap_results['true_optimal_value']}")
print(f"xhat true value: {true_gap_results['xhat_true_value']}")
print(f"True optimality gap: {true_gap_results['true_gap']}")

INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP



True finite-population HF quantities:
True optimal value: 58593.92432766313
xhat true value: 63780.21665537743
True optimality gap: 5186.292327714298
